In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

train = pd.read_parquet("../data/processed/train.parquet").drop(columns=["target"])

typical = {}
for col in train.columns:
    values = train[col]
    if values.isna().mean() > 0.5:
        typical[col] = np.nan
    elif pd.api.types.is_numeric_dtype(values) or pd.api.types.is_datetime64_any_dtype(values):
        typical[col] = values.median()
    else:
        typical[col] = values.mode().iloc[0]

typical_applicant = pd.DataFrame([typical]).astype(train.dtypes.to_dict())
typical_applicant.to_parquet("../models/typical_applicant.parquet", index=False)

print("Types match training:", (typical_applicant.dtypes == train.dtypes).all())

Types match training: True


In [2]:
test = pd.read_parquet("../data/processed/test.parquet")
demo_loans = test.sample(n=2000, random_state=7)
demo_loans.to_parquet("../models/demo_loans.parquet", index=False)

print(demo_loans.shape)
print("Default rate in demo loans:", round(demo_loans["target"].mean(), 4))

(2000, 72)
Default rate in demo loans: 0.1455


In [3]:
import sys
sys.path.append("..")

import joblib
import pandas as pd

from src.app_support import build_applicant
from src.policy import load_policy, decide

model = joblib.load("../models/lightgbm_v2_calibrated.joblib")
policy = load_policy("../results/policy.json")
typical = pd.read_parquet("../models/typical_applicant.parquet")

form = {
    "loan_amnt": 15000, "purpose": "debt_consolidation", "annual_inc": 55000,
    "emp_length": "5 years", "home_ownership": "RENT", "addr_state": "CA",
    "fico": 690, "dti": 22.5, "acc_open_past_24mths": 6,
    "inq_last_6mths": 2, "revol_util": 68, "credit_years": 9,
}

for label, row in [("Typical applicant", typical),
                   ("Form example", build_applicant(typical, form, pd.Timestamp.today()))]:
    p = model.predict_proba(row)[:, 1][0]
    print(f"{label}: {p * 100:.1f}% → {decide(p, policy['cutoff'], policy['review_margin'])}")

Typical applicant: 8.4% → Approve
Form example: 18.5% → Approve
